# Lesson 12 — Production hardening and the inspector

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/12-production-hardening/colab.ipynb)

*Lessons 01–11 improved what the system finds. This one is about running it: refusing safely, caching, citing, costing, and seeing inside it.*

**What you will learn**

- Calibrating a refusal threshold from data — and why the stronger ranker is the weaker calibration signal
- Prompt caching as an exact prefix match, and how it silently breaks
- Two kinds of citation, and what each can actually guarantee
- Cost and token accounting: judge cost per completed task
- Observability: running the HTTP service inside Colab and building a retrieval inspector
- Reading the whole course as one results table

| | |
|---|---|
| ⏱️ Time | 60–75 min |
| 🖥️ Runtime | CPU is fine |
| 🤖 Needs the generator (Ollama) | yes — for some cells |

**Before this:** [11-graphrag](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/11-graphrag/colab.ipynb)

> **How to use this notebook.** Run the cells top to bottom (`Shift+Enter`). Every
> time you see **🔮 Predict**, stop and write down what you expect *before* running
> the next cell — the gap between prediction and result is where the learning is.
> Answers to the check-your-understanding questions are hidden behind ▶ toggles.

## 0. Setup

This cell clones the course repo into the Colab runtime, installs the
dependencies, and (if `USE_LLM` is ticked) installs and starts
[Ollama](https://ollama.com) with the small `qwen2.5:1.5b` generator. It is safe
to re-run. Everything in the repo's `ragkit/` library becomes importable.

If an import fails after the install, use **Runtime → Restart session** and run
this cell again.

In [ ]:
# @title ⚙️ Setup — run me first
USE_LLM = True  # @param {type:"boolean"}
BRANCH = "main"  # @param {type:"string"}

import subprocess
import sys
from pathlib import Path


def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "ragkit").is_dir() and (p / "colab" / "colab_setup.py").exists():
            return p
    return None


REPO = _find_repo()
if REPO is None:
    REPO = Path("/content/rag-from-scratch-to-production")
    if not REPO.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                BRANCH,
                "https://github.com/safar-byte/rag-from-scratch-to-production.git",
                str(REPO),
            ],
            check=True,
        )
sys.path.insert(0, str(REPO / "colab"))
from colab_setup import compare_runs, eval_frames, results_frame, setup  # noqa: F401

LLM_READY = setup(llm=USE_LLM)

## 1. Refusing safely — the end of a thread that started in lesson 03

| Mechanism | Correct refusal | False refusals | Cost | Where |
|---|---|---|---|---|
| default prompt | 0.300 | 0/5 | — | baseline |
| **strict prompt** | **0.900** | 1/5 | free | `generate/prompt.py` |
| score floor 0.60 | 0.500 | 0/34 | one comparison | `generate/abstain.py` |
| CRAG grading | reads the passages | measured in lesson 10 | one model call | `agent/grade.py` |

The score floor refuses *before* any model call if the best retrieved chunk is not
similar enough to the question. It is the cheapest lever — but only if the threshold is
**calibrated, not guessed**. Let's calibrate it.

In [ ]:
import numpy as np
import pandas as pd

from ragkit.eval.golden import load_golden
from ragkit.generate import DEFAULT_SCORE_FLOOR, abstain
from ragkit.pipeline import RagPipeline

pipeline = RagPipeline(strategy="rerank")
if pipeline.count() == 0:
    print(pipeline.ingest())

rows = []
for q in load_golden():
    results = pipeline.retrieve(q.question, top_k=5)
    rows.append(
        {
            "question": q.question,
            "answerable": q.kind != "unanswerable",
            "best dense": abstain.best_retrieval_score(results),
            "best cross-encoder": max(s.scores.get("rerank", -99) for s in results),
        }
    )
scores = pd.DataFrame(rows)
scores.groupby("answerable")[["best dense", "best cross-encoder"]].describe().T.round(3)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for ax, col in zip(axes, ["best dense", "best cross-encoder"], strict=True):
    for answerable, colour in [(True, "#4C72B0"), (False, "#C44E52")]:
        subset = scores[scores.answerable == answerable][col]
        ax.scatter(
            subset,
            np.random.default_rng(0).uniform(0, 1, len(subset)) + (0 if answerable else 1.2),
            color=colour,
            label="answerable" if answerable else "unanswerable",
            alpha=0.8,
        )
    ax.set_title(col)
    ax.set_yticks([])
    ax.legend(loc="upper left")
axes[0].axvline(DEFAULT_SCORE_FLOOR, color="grey", ls="--")
plt.suptitle("Can a threshold separate red from blue?")
plt.show()

A threshold works only where the red dots (unanswerable) sit *left of* every blue dot
(answerable). Sweep it and count both kinds of error:

In [ ]:
def sweep(column, thresholds):
    ans, una = scores[scores.answerable], scores[~scores.answerable]
    return pd.DataFrame(
        [
            {
                "threshold": t,
                "unanswerable caught": f"{(una[column] < t).sum()}/{len(una)}",
                "answerable wrongly refused": f"{(ans[column] < t).sum()}/{len(ans)}",
            }
            for t in thresholds
        ]
    ).set_index("threshold")


sweep("best dense", [0.50, 0.55, 0.60, 0.63, 0.65, 0.70])

**The calibration rule:** pick the highest threshold that refuses *no* valid question
(if a wrong answer and a missing answer cost the same to you — otherwise weigh them).
On the repo's run that was **0.60**: half the unanswerable questions caught for free,
and past ~0.63 it starts rejecting real questions faster than it catches fakes.

**The ones it cannot catch are the dangerous ones.** Questions about settings that do
not exist score *higher* than many genuine questions, because they are about the same
topic. That is the method's ceiling, not a tuning failure — a similarity score measures
topical closeness, not answerability (lesson 10).

### The stronger ranker is the weaker calibration signal

The gate deliberately reads the **dense** score even when the pipeline reranks. Sweep
the cross-encoder score instead:

In [ ]:
ans, una = scores[scores.answerable], scores[~scores.answerable]
print(f"lowest answerable CE score  : {ans['best cross-encoder'].min():.2f}")
print(f"highest unanswerable CE score: {una['best cross-encoder'].max():.2f}")
sweep("best cross-encoder", [-8, -6, -4, -2, 0])

The cross-encoder's score ranges overlap far worse (the repo found lowest answerable
−9.80 against highest unanswerable −4.16): any threshold rejects many more real
questions than it catches fakes. **Ranking quality and calibration are different
properties**, and a reranker is trained only for the first. Recalibrate the floor for
any other embedding model or corpus — cosine scores from a different model live on a
different scale entirely.

The gate in action:

In [ ]:
!{sys.executable} lessons/12-production-hardening/run.py

## 2. Prompt caching — a prefix match that breaks silently

On a paid API, a provider can cache the processed form of a prompt **prefix** and bill
later requests that share it at ~10% of the input price. The rules (Claude):

- The prompt is rendered in a fixed order: `tools` → `system` → `messages`.
- Caching is an **exact byte-prefix match**. Any change anywhere in the prefix
  invalidates everything after it.
- So: **stable content first, the user's question last** — which is exactly why
  `build_prompt` puts the context before the question, a decision made back in lesson 01.

You can reason about this without an API. Here are two requests and the length of the
prefix they share:

In [ ]:
import os

from ragkit.generate import SYSTEM_PROMPT, build_prompt


def shared_prefix(a: str, b: str) -> int:
    return len(os.path.commonprefix([a, b]))


ctx = pipeline.retrieve("What is reciprocal rank fusion?", top_k=3)
good_a = SYSTEM_PROMPT + build_prompt("What is reciprocal rank fusion?", ctx)
good_b = SYSTEM_PROMPT + build_prompt("Why does RRF use ranks?", ctx)

# The classic mistake: something that varies per request, placed early.
bad_a = f"Request id: 7f3a · {SYSTEM_PROMPT}" + build_prompt("What is reciprocal rank fusion?", ctx)
bad_b = f"Request id: 91c2 · {SYSTEM_PROMPT}" + build_prompt("Why does RRF use ranks?", ctx)

print(
    f"stable prefix, question last : {shared_prefix(good_a, good_b):5d} of {len(good_a)} chars cacheable"
)
print(
    f"request id in the system text: {shared_prefix(bad_a, bad_b):5d} of {len(bad_a)} chars cacheable"
)

A timestamp, a request id, an unsorted JSON object, a tool list built from a `set` —
any of them in the prefix and the cache never hits. **Nothing errors; you just pay full
price.** The only defence is verification: assert `usage.cache_read_input_tokens > 0`
on a repeated request. (Also: prefixes below the model's *minimum cacheable length* are
never cached.)

### Optional: verify it on the cloud profile

If you have API keys, add `ANTHROPIC_API_KEY` and `VOYAGE_API_KEY` to Colab's
**Secrets** panel (🔑 in the left sidebar) and run the next cell. It makes two calls
with the same long system prompt and prints the cache counters. Without keys it skips.

In [ ]:
try:
    from google.colab import userdata

    keys = {k: userdata.get(k) for k in ("ANTHROPIC_API_KEY", "VOYAGE_API_KEY")}
except Exception:
    keys = {k: os.environ.get(k) for k in ("ANTHROPIC_API_KEY", "VOYAGE_API_KEY")}

if all(keys.values()):
    import subprocess

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "anthropic", "voyageai"], check=True
    )
    from ragkit.config import Profile, Settings
    from ragkit.providers.cloud import ClaudeGenerator

    cloud = ClaudeGenerator(Settings(RAG_PROFILE=Profile.CLOUD, **keys), cheap=True)
    long_system = SYSTEM_PROMPT + "\n\n" + "\n\n".join(s.chunk.text for s in ctx) * 4
    for attempt in (1, 2):
        text, usage = cloud.generate(
            system=long_system, prompt="What is RRF?", max_tokens=200, cache_system=True
        )
        print(
            f"call {attempt}: cache_write={usage.cache_write_tokens} "
            f"cache_read={usage.cache_read_tokens} cost=${usage.cost_usd:.5f}"
        )
    print("call 2 must show cache_read > 0 — otherwise something in the prefix varies.")
else:
    print("No cloud keys found — skipping. Everything else in this notebook runs offline.")

## 3. Citations — two mechanisms, not equal

**Marker parsing** (`generate/answer.py`) asks the model to type `[1]`, `[2]` and
resolves those numbers back to chunks. It is the fallback, and it is unreliable *by
construction*: the model can cite a passage it did not use, or use one it did not cite,
and nothing can detect either.

In [ ]:
from ragkit.generate import extract_citations

for answer_text in [
    "RRF sums 1/(k + rank) across retrievers [1][3].",
    "RRF uses ranks because scores are not comparable [2, 9].",  # [9] does not exist
    "RRF was invented in 2009 by Cormack et al. [1].",  # cited, but is it in [1]?
]:
    cites = extract_citations(answer_text, ctx)
    print(f"{answer_text}\n   → resolved to: {[c.doc_id for c in cites]}")

The third answer cites `[1]` for a claim that may not be in passage 1 at all — and the
citation resolves happily. Marker parsing records what the model *typed*, not what it
*grounded*.

**The Citations API** (cloud) returns the exact spans the model grounded each claim in,
with character offsets into the source document — a different guarantee entirely.
`ClaudeGenerator.generate_with_citations` implements it. Two constraints worth knowing:
it is **incompatible with structured output** (`output_config.format` → 400), so the
citation path and the JSON path stay separate; and citations must be enabled on every
document block or none.

This is also why `Chunk.text` and `Chunk.context` stayed separate in lesson 07: cite
generated text and the system can no longer prove anything it says.

## 4. Cost accounting

Every `Answer` carries a `Usage`: input, output, cache-read and cache-write tokens plus
a dollar figure. The local profile reports `$0.00` honestly. Here is what the same
pipeline would cost on the cloud profile, from real prompt sizes:

In [ ]:
import tiktoken

from ragkit.providers.cloud import PRICES_USD_PER_MTOK

enc = tiktoken.get_encoding("cl100k_base")  # a proxy for Claude's tokenizer
prompt_tokens = np.mean(
    [
        len(
            enc.encode(
                SYSTEM_PROMPT + build_prompt(q.question, pipeline.retrieve(q.question, top_k=5))
            )
        )
        for q in load_golden()[:15]
    ]
)
OUT = 150
rows = []
for model, (pin, pout) in PRICES_USD_PER_MTOK.items():
    fresh = (prompt_tokens * pin + OUT * pout) / 1e6
    rows.append(
        {
            "model": model,
            "prompt tokens": int(prompt_tokens),
            "$ per query": fresh,
            "$ per 1M queries": fresh * 1e6,
        }
    )
pd.DataFrame(rows).round(5)

Two habits:

- **Judge cost per completed task, not per request.** A cheaper model that needs two
  retries and a clarifying turn is not cheaper. CRAG's extra grading call (lesson 10)
  doubles calls per question — that is the number to put next to its refusal rate.
- **Record cost next to quality, always.** "Better" that costs 40× more is a trade-off,
  not a win, and a results table without a cost column cannot tell you which you have.

## 5. Observability: run the service and inspect it

Almost every bad RAG answer is a bad retrieval, and that is invisible from the final
text. Every stage's scores stay attached to each chunk through the whole pipeline —
`dense_rank`, `bm25_rank`, the RRF votes, `rank_before_rerank`, the cross-encoder score —
and the HTTP service returns all of it.

We can run the real FastAPI service **inside this Colab runtime**, in a background
thread:

In [ ]:
import threading
import time

import httpx
import uvicorn

from ragkit.api.main import app

if "server" not in globals():
    server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning"))
    threading.Thread(target=server.run, daemon=True).start()
    for _ in range(50):
        try:
            if httpx.get("http://127.0.0.1:8000/health", timeout=1).status_code == 200:
                break
        except httpx.TransportError:
            time.sleep(0.2)

api = httpx.Client(base_url="http://127.0.0.1:8000", timeout=300)
api.get("/health").json()

In [ ]:
try:
    from google.colab.output import eval_js

    print("Interactive API docs:", eval_js("google.colab.kernel.proxyPort(8000)") + "docs")
except Exception:
    print("Interactive API docs: http://127.0.0.1:8000/docs")

`POST /retrieve` skips the model entirely, so once the models are warm it answers in
well under a second. It is the endpoint you use most when debugging. Let's build a
small **inspector** on top of it — the same idea as the repo's React UI (`ui/`): every
result rendered as its whole history, with a badge for how the reranker moved it.

In [ ]:
from html import escape

from IPython.display import HTML


def inspect(question, strategy="rerank", top_k=5):
    data = api.post(
        "/retrieve", json={"question": question, "strategy": strategy, "top_k": top_k}
    ).json()
    cards = []
    for c in data["contexts"]:
        s = c["scores"]
        before = s.get("rank_before_rerank")
        if before is None:
            badge = ""
        elif before > c["rank"]:
            badge = f"<span style='color:#2e7d32'>▲ up {int(before)}→{c['rank']}</span>"
        elif before < c["rank"]:
            badge = f"<span style='color:#c62828'>▼ down {int(before)}→{c['rank']}</span>"
        else:
            badge = "<span style='color:#777'>● unmoved</span>"
        chips = " · ".join(f"{escape(k)} <b>{v:g}</b>" for k, v in s.items())
        cards.append(
            f"<div style='border:1px solid #ccc;border-radius:6px;padding:8px;margin:6px 0'>"
            f"<b>#{c['rank']} {escape(c['doc_id'])}</b> &nbsp; {badge}<br>"
            f"<small style='color:#555'>{chips}</small>"
            f"<div style='margin-top:4px'>{escape(c['text'][:260])}…</div></div>"
        )
    return HTML(
        f"<h4>{escape(question)}</h4><small>{escape(data['retriever'])} · "
        f"{data['elapsed_ms']} ms</small>" + "".join(cards)
    )


inspect("Why do I get charged twice?")

In [ ]:
inspect("What does exit status 75 mean?", strategy="router")

The full `/query` endpoint returns the answer *plus* the assembled prompt, token usage
and timing trace — everything needed to answer "why did it say that?" without
re-running anything:

In [ ]:
if LLM_READY:
    r = api.post(
        "/query",
        json={
            "question": "What is the default value of SHARD_REPLICATION_FACTOR?",
            "strategy": "rerank",
            "score_floor": 0.60,
            "strict_prompt": True,
        },
    ).json()
    print("answer   :", r["answer"])
    print("abstained:", r["abstained"])
    print("usage    :", r["usage"])
    print("trace    :", {k: v for k, v in r["trace"].items() if not isinstance(v, list)})
    print("\nprompt (first 400 chars):\n", r["prompt"][:400])

### Three diagnostic patterns, mechanised

| What you see in the trace | Diagnosis | Fix |
|---|---|---|
| correct chunk absent at any depth | retrieval failed | chunking, hybrid, contextual — reranking is irrelevant |
| correct chunk at rank 40, shortlist 25 | shortlist too shallow | deepen the shortlist (`first_relevant_rank` tells you) |
| correct chunk reached the prompt, answer still wrong | generation | prompt or model |

## 6. Index versioning

`ingest` resets the index by default (lesson 01): partially re-indexing a corpus whose
chunk boundaries moved leaves orphans, and a stale index is much harder to debug than a
slow rebuild. For a real system, `data/07-operations.md` describes the zero-downtime
arrangement: an `INDEX_GENERATION` counter, queries pinned to the generation current
when they started, and the old generation retained until in-flight queries drain. (The
corpus this repo searches has been describing the technique the repo needs all along.)

**Not deployable as-is**, and worth saying plainly: CORS allows localhost only, there is
no auth, no rate limiting, and no index versioning. Exercise 3 walks through it.

## 7. The end of the course: read the table

Go back to `benchmarks/results.md` and read it as one story. That is what the course was
actually for: not thirteen techniques, but the habit of asking what each one was worth —
and being willing to write down the answer when it was "nothing".

In [ ]:
from IPython.display import Markdown

Markdown(Path("benchmarks/results.md").read_text())

| Technique | What it was worth *here* |
|---|---|
| Evaluation harness (03) | everything — it caught every other result in this table |
| Hybrid search (04) | lost to dense as a ranker; won as a candidate pool |
| Reranking (05) | the biggest retrieval gain; shortlist 10 beat the conventional 25 |
| Query rewrite (06) | worse on every metric at 2.2× latency |
| Contextual retrieval (07) | helped dense; hurt the reranker until it read source text only |
| Parent-document (08) | more diverse context for the generator; recall unchanged |
| Routing (09) | small, correct gain, from 4 of 44 questions |
| CRAG (10) | with a 1.5B grader, refused 8 of 10 answerable questions |
| GraphRAG (11) | the wrong shape for this corpus (mean degree 1.6) |
| Strict prompt (03/12) | the largest refusal lever — and free |

## 8. Check your understanding

**1. Why does the abstention gate read the dense score even when the pipeline reranks?**

<details><summary>▶ Show answer</summary>

Because cross-encoder scores separate answerable from unanswerable questions far
worse than cosine does on this corpus. Rerankers are trained to *order* passages,
not to produce calibrated "is there an answer" scores.

</details>

**2. Your cloud bill did not drop after enabling prompt caching. What do you check?**

<details><summary>▶ Show answer</summary>

`cache_read_input_tokens` on repeated requests. If it is zero, something in the
prefix varies between requests (timestamp, request id, ordering), the variable part
is placed before the stable part, or the prefix is below the minimum cacheable
length.

</details>

**3. What can marker-parsed citations *not* guarantee?**

<details><summary>▶ Show answer</summary>

That the cited passage actually supports the claim, or that every passage used was
cited. They record the numbers the model typed. The Citations API returns spans the
model actually grounded in.

</details>

## ✍️ Exercises

1. **Recalibrate** the score floor for a different embedding model (e.g.
   `BAAI/bge-base-en-v1.5` via `Settings(local_embedding_model=...)`, re-ingested into a
   `/tmp` index). Is 0.60 still right?
2. **Diagnose three failures from the trace alone.** Pick three golden questions that
   miss at k=5 and classify each with the table in section 5 using only
   `inspect(...)`.
3. **Ship-readiness review.** List what you would add before exposing `ragkit.api` to
   the internet: auth, rate limiting, CORS, index versioning, timeouts — and where in the
   code each belongs.

## Key takeaways

1. Calibrate refusal thresholds from data; the best ranker is not the best calibration signal.
2. Prompt caching is an exact prefix match — stable content first, and verify cache reads.
3. Marker citations record what the model typed; the Citations API records what it grounded.
4. Report cost next to quality, per completed task.
5. Keep every stage's scores on every chunk; an inspector over `/retrieve` makes retrieval bugs visible.
6. Measure first, optimise second — and keep the negative results.